# PanAf Ape Detection — Phase 1 ("See")

**Pretrained MegaDetector V6 inference over PanAf500 great-ape camera-trap clips.**

This notebook is a **scaffold**. The detection, tracking and annotation stages are *not implemented
yet* — the sections below are placeholders that describe what each stage will do and where its code
will live once `panaf_ape_detection` grows those modules.

Nothing in this notebook has produced a detection. Do not treat any section as evidence of results.

### Ground rules

1. **Business logic does not live in this notebook.** Every reusable operation belongs in
   `src/panaf_ape_detection/` with a test. The notebook orchestrates and displays; it does not
   define. A cell that grows past a few lines of glue is a module that has not been written yet.
2. **Do not download the full PanAf20K dataset here.** Phase 1 uses ~5–10 PanAf500 clips.
3. **Do not commit this notebook with output cells.** Stored output can embed dataset frames, which
   is a licensing problem as well as a repository-hygiene one. `scripts/verify_repository.py`
   enforces this.
4. **Clear outputs before committing:** *Edit → Clear all outputs*.

## 0. Select a GPU runtime

**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.**

Do this **before** installing anything: changing the runtime type restarts the session and discards
everything installed so far.

CPU inference over video is slow enough to be impractical for this phase. The cell below reports
what you actually got — if it shows no GPU, fix the runtime before continuing.

In [ ]:
# Report the runtime GPU, if any. Safe to run on a CPU-only runtime.
!nvidia-smi || echo "No GPU detected — go to Runtime > Change runtime type and select a GPU."

## 1. Clone the repository

Replace `<YOUR-FORK-URL>` with your repository URL. If the repository is private, use a
personal access token or the GitHub CLI — **never paste a token into a notebook cell you might
commit.**

In [ ]:
REPO_URL = "<YOUR-FORK-URL>"   # e.g. https://github.com/<user>/panaf-ape-detection.git
REPO_DIR = "/content/panaf-ape-detection"

import os
from pathlib import Path

if not Path(REPO_DIR).exists():
    !git clone $REPO_URL $REPO_DIR
else:
    print(f"{REPO_DIR} already exists; skipping clone.")

os.chdir(REPO_DIR)
# Make repository-root discovery unambiguous for the package.
os.environ["PANAF_REPO_ROOT"] = REPO_DIR
print("Working directory:", Path.cwd())

## 2. Install the environment

Installed from `requirements-colab.txt`, which is **generated from `uv.lock`** — not maintained
separately. That is what makes the Colab environment match the locked local one instead of drifting
with whatever Colab happens to preinstall.

To regenerate it locally after changing dependencies:

```bash
uv export --extra inference --no-hashes --no-dev --format requirements-txt -o requirements-colab.txt
```

This installs a large dependency tree (PyTorch, ultralytics, lightning, gradio) and takes several
minutes. **Colab may ask you to restart the session afterwards — do it**, then continue from
section 3 without re-running section 2.

In [ ]:
# Install the locked inference environment, then the project package itself.
!pip install --quiet -r requirements-colab.txt
!pip install --quiet -e . --no-deps

## 3. Verify the environment

Confirms the package imports, the repository structure is intact, and reports whether a GPU is
visible. **No model weights are downloaded by this step.**

If `doctor` reports the inference stack as missing right after section 2, the session probably needs
the restart Colab suggested.

In [ ]:
!python scripts/check_environment.py

In [ ]:
!python -m panaf_ape_detection.cli doctor

In [ ]:
!python scripts/verify_repository.py

# And, once the inference extra is installed, confirm the heavy stack actually
# works -- imports, ByteTrack, NumPy interop, video round-trip. Downloads no weights.
!python scripts/smoke_inference.py

## 4. (Optional) Mount Google Drive

**Optional and separable.** Skip this entirely if you are working with files uploaded directly to
the Colab session.

Useful because Colab sessions are ephemeral: Drive gives you somewhere for the dataset clips to
survive a disconnect, and somewhere to persist generated artifacts.

Mounting grants this notebook access to your entire Drive. Review the permission prompt.

**Dataset licensing still applies to files on Drive.** Do not place PanAf files in a shared or
public Drive folder — that is redistribution. See `docs/obsidian/05 Technical/licensing.md`.

In [ ]:
USE_DRIVE = False   # set to True to mount

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
else:
    print("Skipping Drive mount. Set USE_DRIVE = True to enable.")

## 5. Set up the data folders

Creates the expected local layout. **It does not download anything** — dataset acquisition is
manual and deliberate.

Read `data/README.md` before proceeding. In summary:

- Obtain PanAf500 from the Bristol deposit under its own (non-commercial) licence:
  <https://data.bris.ac.uk/data/dataset/1h73erszj3ckn2qjwm4sqmr2wt>
- Copy only your ~5–10 selected clips into `data/raw/panaf500/`.
- Treat `data/raw/` as immutable. Extracted frames go to `data/interim/`.
- Record your selection and SHA-256 checksums in `data/sample_manifest.csv`.

In [ ]:
from pathlib import Path

for relative in [
    "data/raw/panaf500/videos",
    "data/raw/panaf500/annotations",
    "data/interim/frames",
    "data/processed",
]:
    Path(relative).mkdir(parents=True, exist_ok=True)
    print("ready:", relative)

manifest = Path("data/sample_manifest.csv")
if not manifest.exists():
    print(
        "\nNo data/sample_manifest.csv yet. Copy the template and fill it in:\n"
        "  cp data/sample_manifest.example.csv data/sample_manifest.csv"
    )

In [ ]:
# Compute SHA-256 digests for the manifest. `provenance.file_sha256` is the same
# function the pipeline will use, so the digests you record here and the ones a
# run verifies against cannot disagree.
from pathlib import Path

from panaf_ape_detection.provenance import file_sha256

videos = [p for p in sorted(Path("data/raw/panaf500/videos").glob("*")) if p.is_file()]

if videos:
    for video in videos:
        print(f"{video.name}  {file_sha256(video)}")
else:
    print("(no videos found — place your selected clips in data/raw/panaf500/videos/)")

## 6. Load and inspect the configuration

Demonstrates the typed configuration system, which **is** implemented. Relative paths resolve
against the repository root, unknown keys are rejected, and ranges are validated.

`configs/colab.yaml` differs from `configs/base.yaml`: `device: cuda`, a larger `frame_stride` and
a smaller `max_clips`, so a first pass finishes inside a session.

In [ ]:
from panaf_ape_detection.config import load_config

config = load_config("configs/colab.yaml")

for key, value in config.describe().items():
    print(f"{key:32} {value}")

In [ ]:
# Paths are resolved absolutely, from the repository root.
print("raw data:  ", config.paths.raw_data_dir)
print("interim:   ", config.paths.interim_data_dir)
print("artifacts: ", config.paths.artifacts_dir)
print("manifest:  ", config.data.manifest_path)
print()
print("model:     ", config.model.model_name, "/", config.model.variant)
print("threshold: ", config.model.confidence_threshold)
print("recognised variant:", config.model.variant_is_recognised)

---

# Pipeline sections — NOT IMPLEMENTED

Everything below is a **placeholder**. The functions referenced do not exist yet. Each section
records what the stage will do, so the implementation has a target and this notebook can be filled
in stage by stage.

**Do not fabricate output in these cells.** An empty placeholder is honest; a hand-written
"example detection" is not.

## 6b. Resolve the device — and **verify** it

> **Read this cell even if you skip the rest.** It is the difference between a
> GPU run and a GPU-priced CPU run.

PyTorch-Wildlife 1.3.0 accepts a `device=` argument, stores it, and **never applies it** — the line
that would is commented out in `yolov8_base._load_model`. The weights load on CPU, nothing raises,
and `detector.device` still reports `"cuda"`.

On Colab that is the worst case: you select a T4, everything claims CUDA, and inference crawls at
CPU speed while the run metadata records a device the model never touched.

So: resolve the device with `runtime.resolve_device`, then **check the tensors** with
`runtime.module_device` after the model is built. Full detail in `docs/obsidian/05 Technical/model.md`.

In [ ]:
from panaf_ape_detection.runtime import available_devices, resolve_device, set_seeds

# Seed everything before anything stochastic happens.
set_seeds(config.project.seed)
print("seeded with", config.project.seed)

print("available devices:", sorted(d.value for d in available_devices()))

# `auto` prefers cuda -> mps -> cpu. An explicit device that is unavailable
# raises rather than silently falling back, because a silent CPU fallback makes
# every timing number in the write-up meaningless.
device = resolve_device(config.model.device)
print(f"configured {config.model.device.value!r} -> resolved {device.value!r}")

if device.value == "cpu":
    print(
        "\nWARNING: running on CPU. If you selected a GPU runtime, this is wrong.\n"
        "  Runtime > Change runtime type > T4 GPU, then Runtime > Restart session."
    )

In [ ]:
# The verification helper, ready for when a detector exists (section 8).
#
# After constructing the model, ALWAYS run:
#
#     from panaf_ape_detection.runtime import module_device
#     actual = module_device(detector)
#     if actual is None or not actual.startswith(device.value):
#         import torch
#         torch_device = torch.device(device.value)
#         detector.predictor.model.to(torch_device)   # weights
#         detector.predictor.device = torch_device    # inputs -- omit and the
#                                                     # forward pass crashes
#         detector.predictor.args.device = device.value
#
# and record `module_device(detector)` in run metadata, never the requested value.
#
# `scripts/smoke_detect.py` is the working reference implementation.

from panaf_ape_detection.runtime import module_device

print("module_device() on a non-model returns:", module_device(object()))
print("(None means 'could not determine' -- never treat that as success)")

## 7. Frame extraction — *not implemented*

**Will do:** decode each manifest clip, honour `data.frame_stride`, and write frames to
`data/interim/frames/<clip_id>/`, recording frame count, resolution and fps.

**Will live in:** `panaf_ape_detection.data.video`

**Notes:** raw clips are never modified; frames are derived data. Verify manifest checksums before
decoding.

In [ ]:
# Planned:
# from panaf_ape_detection.data.video import extract_frames
# frames = extract_frames(config)

raise NotImplementedError(
    "Frame extraction is not implemented yet. See docs/obsidian/05 Technical/architecture.md "
    "(planned module: panaf_ape_detection.data.video)."
)

## 8. MegaDetector V6 inference — *not implemented*

**Will do:** load the configured variant through PyTorch-Wildlife, run detection frame by frame,
and emit `Detection` records to `artifacts/detections/`.

**Will live in:** `panaf_ape_detection.inference.megadetector`

**Notes:**

- The variant comes from config and is **never** left to the upstream default — PyTorch-Wildlife
  1.3.0's defaults (`yolov9c`, `MDV6-rtdetr-x-apache`) are not accepted by their own validation and
  raise `ValueError`. See `docs/obsidian/05 Technical/model.md`.
- Weights download from Zenodo on first use, into the session's cache. They are large; expect a wait.
- MegaDetector emits `animal` / `person` / `vehicle` only. **Not species. Not behaviour.**

In [ ]:
# Planned:
# from panaf_ape_detection.inference.megadetector import MegaDetectorV6Runner
# detector = MegaDetectorV6Runner.from_config(config)   # downloads weights on first use
# detections = detector.run(frames)

raise NotImplementedError(
    "MegaDetector inference is not implemented yet. See docs/obsidian/05 Technical/architecture.md "
    "(planned module: panaf_ape_detection.inference.megadetector)."
)

## 9. Tracking — *not implemented*

**Will do:** associate detections across frames into identity tracks, dropping tracks shorter than
`tracking.minimum_track_length`.

**Will live in:** `panaf_ape_detection.tracking`

**Notes:** the backend is deliberately undecided (`tracking.enabled: false` by default). It will be
chosen from Phase 1c evidence about how stable detections actually are. `supervision` — already
present as a PyTorch-Wildlife dependency — provides ByteTrack.

In [ ]:
# Planned:
# from panaf_ape_detection.tracking import build_tracker
# tracker = build_tracker(config)
# tracks = tracker.run(detections)

raise NotImplementedError(
    "Tracking is not implemented yet; the backend has not been selected. "
    "See docs/obsidian/05 Technical/architecture.md."
)

## 10. Behaviour-label overlay — *not implemented*

**Will do:** read PanAf500's frame-wise behaviour annotations and attach them to tracks for display.

**Will live in:** `panaf_ape_detection.data.annotations` + `panaf_ape_detection.visualization.overlays`

**Notes:** behaviour labels are **dataset ground truth**, never model predictions. Any rendered
output must make that distinction visible — a box is a prediction, a behaviour label is not.

In [ ]:
# Planned:
# from panaf_ape_detection.data.annotations import load_behavior_labels
# labels = load_behavior_labels(config)
# annotated = attach_behavior_labels(tracks, labels)

raise NotImplementedError(
    "Behaviour-label overlay is not implemented yet. See docs/obsidian/05 Technical/architecture.md."
)

## 11. Video export — *not implemented*

**Will do:** render boxes, scores, track ids and behaviour labels onto frames and encode annotated
clips (and GIFs) to `artifacts/videos/`, honouring the `video` config section.

**Will live in:** `panaf_ape_detection.visualization.video`

**Notes:** requires FFmpeg. Exported clips are derived works of the dataset — check the licence
before sharing them.

In [ ]:
# Planned:
# from panaf_ape_detection.visualization.video import export_annotated_clip
# output = export_annotated_clip(config, annotated)

raise NotImplementedError(
    "Video export is not implemented yet. See docs/obsidian/05 Technical/architecture.md."
)

## 12. Qualitative evaluation — *not implemented*

**Will do:** support structured review of failure cases — missed detections, false positives, ID
switches — tabulated against clip conditions.

**Will live in:** `panaf_ape_detection.evaluation.qualitative`

**Notes:** Phase 1 is qualitative by design. A scored comparison against PanAf500 ground-truth boxes
is Phase 2 and needs matching criteria decided in advance. Findings go into
`reports/phase1_writeup_template.md` — copied to a dated file first, never invented.

In [ ]:
# Planned:
# from panaf_ape_detection.evaluation.qualitative import summarize_failures
# summary = summarize_failures(config, annotated)

raise NotImplementedError(
    "Qualitative evaluation is not implemented yet. See docs/obsidian/05 Technical/architecture.md."
)

---

## 13. Save outputs and run metadata

Colab sessions are ephemeral: **anything not copied out is lost when the session ends.**

Once inference exists, every run will write a `RunMetadata` record to `artifacts/metadata/`,
capturing the git commit and dirty flag, resolved config, dependency versions, device, model variant,
confidence threshold, seed, input checksums, output paths and elapsed time. Schema:
`src/panaf_ape_detection/types.py`. See `docs/obsidian/05 Technical/reproducibility.md`.

**Nothing writes that record today.**

Reminders before you finish a session:

- `artifacts/` is git-ignored — copy anything you want to keep to Drive.
- **Do not commit dataset files, frames, weights, or annotated clips.**
- **Clear all outputs before committing this notebook** (*Edit → Clear all outputs*).
- Write an entry in `experiments/experiment_log.md`, including anything that failed.

In [ ]:
from pathlib import Path

artifacts = Path(config.paths.artifacts_dir)
if artifacts.exists():
    produced = sorted(p for p in artifacts.rglob("*") if p.is_file())
    print(f"{len(produced)} artifact file(s) in {artifacts}")
    for path in produced[:20]:
        print("  ", path.relative_to(artifacts))
else:
    print(f"No artifacts directory yet at {artifacts} — nothing has been generated.")

In [ ]:
# Build a real run-metadata record. This is not a placeholder: `build_run_metadata`
# is implemented and tested. What does *not* exist yet is a pipeline stage that
# runs inference over clips and writes one automatically.
from panaf_ape_detection.provenance import build_run_metadata, default_metadata_path

metadata = build_run_metadata(config, device=device)

print("experiment: ", metadata.experiment_name)
print("commit:     ", metadata.git_commit, "(dirty)" if metadata.git_dirty else "(clean)")
print("device:     ", metadata.device.value)
print("variant:    ", metadata.model_variant)
print("threshold:  ", metadata.confidence_threshold)
print("seed:       ", metadata.seed)
print("deps:       ", metadata.dependency_versions)
print("\nwould be written to:", default_metadata_path(config))

In [ ]:
# Optional: copy artifacts to Drive so they survive the session.
USE_DRIVE = False
DRIVE_DESTINATION = "/content/drive/MyDrive/panaf-ape-detection/artifacts"

if USE_DRIVE:
    import shutil
    from pathlib import Path

    source = Path(config.paths.artifacts_dir)
    if source.exists():
        shutil.copytree(source, DRIVE_DESTINATION, dirs_exist_ok=True)
        print("Copied artifacts to", DRIVE_DESTINATION)
    else:
        print("Nothing to copy — no artifacts have been generated.")
else:
    print("Drive copy disabled. Set USE_DRIVE = True to enable.")